In [ ]:
import os
os.chdir(r"C:\Users\Annie.LAPTOP-NJ5KV9M1\AGN_mapping")

In [ ]:
pip install astropy scipy matplotlib numpy h5py tqdm numba pint-pulsar emcee corner statsmodels pyfftw tbb

In [ ]:
pip install pytest pytest-astropy jinja2 docutils sphinx-astropy nbsphinx pandoc ipython jupyter notebook towncrier tox black

In [ ]:
pip install stingray PyROA emcee tabulate corner astropy

In [ ]:
import shutil

# Replace '/path/to/work_directory' with your actual work directory

src = r'C:\Users\Annie.LAPTOP-NJ5KV9M1\AGN_mapping\CPython\PYCCF.py'
dst = r'PYCCF.py'
dst2 = r'xcorspc.pyx'

shutil.copy(src, dst)
shutil.copy(src, dst2)

In [ ]:
import os
print(os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import argparse
import random

from scipy import stats
import scipy.integrate as integrate

from stingray import Lightcurve, Crossspectrum, AveragedCrossspectrum

import PYCCF as myccf
import PyROA
import psi

import importlib
importlib.reload(psi)


<module 'psi' from 'C:\\Users\\Annie.LAPTOP-NJ5KV9M1\\AGN_mapping\\psi.py'>

In [ ]:
import scipy
print(scipy.__version__)
print(scipy.__file__)

1.15.2
c:\Users\Annie.LAPTOP-NJ5KV9M1\miniconda3\envs\agn\lib\site-packages\scipy\__init__.py


In [ ]:
#import AGN light curve
directory_path = r'C:\Users\Annie.LAPTOP-NJ5KV9M1\AGN_mapping\data'
file_path = os.path.join(directory_path, 'lc30mins.dat')

column_index1 = 0  
column_index2 = 1

with open(file_path, 'r') as file:
    # Read all lines
    lines = file.readlines()
    data_lines = lines[1:]
    
   # Extract the desired columns from each line and convert to float
    column1_data = [float(line.split()[column_index1]) for line in data_lines]
    column2_data = [float(line.split()[column_index2]) for line in data_lines]

time = np.array(column1_data)/(24*3600)
flux = np.array(column2_data#########################################
##Set Interpolation settings, user-specified
#########################################
lag_range = [-100, 100]  #Time lag range to consider in the CCF (days). Must be small enough that there is some overlap between light curves at that shift (i.e., if the light curves span 80 days, these values must be less than 80 days)
interp = 0.2 #Interpolation time step (days). Must be less than the average cadence of the observations, but too small will introduce noise.
nsim = 10  #Number of Monte Carlo iterations for calculation of uncertainties
mcmode = 0  #Do both FR/RSS sampling (1 = RSS only, 2 = FR only) 
sigmode = 0.2  #Choose the threshold for considering a measurement "significant". sigmode = 0.2 will consider all CCFs with r_max <= 0.2 as "failed". See code for different sigmodes.
#########################################
##Set freq-lag settings, user-specified
#########################################
cs_binning = 100 #averaged cross spectrum binning 
error_in_flux = 0.01 #define percentage error in flux 
SN = 100 #sound to noise ratio 
samplng_rate = 1 #how many samples are taken per day 
sampling_loss = 0.2 # the amount of data lost due to sampling issues
final_length = 350 #length of final array in days 
dt = 1800 #each interval in seconds 
S = 1.0 #SD of BLR function
M = np.log(13.9) # Mean of BLR function
BLR_fraction = 0.5 # BLR fraction

exposure = len(time) #length of full array 
seconds_day = 86400
original_length = exposure/(seconds_day/dt)

times = (np.linspace(-exposure, exposure+1, num = exposure*2)*dt)/(24*3600) # converted to days

#define disk function
def disk(times,wavelength,A1):
    bhmass = 10**7.7       # Msol
    mdot = 0.5         # Msol/yr
    inc = 45           # inclination, degrees
    combi_psi = psi.pytfb_sub(times,1e8,mdot,wavelength,inc,norm=1)
    combi_psi[np.isnan(combi_psi)]=0
    # normalise 
    normi = np.trapz(combi_psi,x=times)
    return A1*(combi_psi/normi)

# Define BLR function
def BLR(times,A2):

    BLR_array = A2/(S*times*np.sqrt(2*np.pi))*np.exp(-(np.log(times)-M)**2/(2*(S**2)))
    BLR_array[np.isnan(BLR_array)]=0
    return  BLR_array

BLR_array_1 = BLR(times, BLR_fraction)
combi_psi_1 = disk(times, 4770, (1 - BLR_fraction))

signal_BLR_Disk = BLR_array_1 + combi_psi_1

### Perform convolution
signal4_BLR = np.convolve(flux, signal_BLR_Disk, mode='full')
signal_BLR = np.convolve(flux, BLR(times, 1), mode='full')
signal_DISK = np.convolve(flux, disk(times, 4770, 1), mode='full')

start_index = int((len(signal4_BLR) - len(flux)) / 2)
end_index = start_index + len(flux)
signal4_BLR = signal4_BLR[start_index:end_index]

start_index1 = int((len(signal_BLR) - len(flux)) / 2)
end_index1 = start_index1 + len(flux)
signal_BLR = signal_BLR[start_index1:end_index1]

start_index2 = int((len(signal_DISK) - len(flux)) / 2)
end_index2 = start_index2 + len(flux)
signal_DISK = signal_DISK[start_index2:end_index2]

#take a smaple each day 
interval = (original_length-final_length)/2
start = int(seconds_day/dt*interval)
end = int(seconds_day/dt*(original_length-interval))

signal_combined = signal4_BLR[start:end]
signal_BLR = signal_BLR[start:end]
signal_DISK = signal_DISK[start:end]
flux = flux[start:end]
time = time[start:end]

lc_BLR = Lightcurve(time,signal_BLR )
lc_full = Lightcurve(time,signal_combined )
lc_disk = Lightcurve(time,signal_DISK )
lc = Lightcurve(time,flux))